# 📑 PageIndex — Vectorless RAG

Reasoning-based RAG with No Vector DB, No Chunking

🔑 Key Concept
Traditional RAG → chunk → embed → cosine similarity → retrieve

+++++++
PageIndex RAG → build tree → LLM reasons over tree → retrieve exact sections

The problem with vector RAG:
Similarity ≠ Relevance
A chunk about "market conditions" may score higher than the actual answer section just because it shares more words with your query.

In [63]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

# Initializes the Groq model using LangChain
llm = ChatGroq(model="openai/gpt-oss-120b")
response = llm.invoke("Hello! How are you ")



In [64]:
# print(response.content)

In [65]:
import os , json, time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("Pageindex Key loaded:", "✅" if PAGEINDEX_API_KEY else "🚫 missing")
print("GROQ_API_KEY Key loaded:", "✅" if GROQ_API_KEY else "🚫 missing")

Pageindex Key loaded: ✅
GROQ_API_KEY Key loaded: ✅


In [66]:
from pageindex import PageIndexClient
from langchain_groq import ChatGroq

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
groq_client = ChatGroq(model="openai/gpt-oss-120b")

In [67]:
PDF_PATH = "./Detailed_Summary.pdf"

print(f"🚀 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)

doc_id = result["doc_id"]

print("✅ Uploaded!")
print(f"📊 Dcoument ID: {doc_id}")
print(" (Save this ID - you 'll use it throughout the notebook)")

🚀 Uploading: ./Detailed_Summary.pdf
✅ Uploaded!
📊 Dcoument ID: pi-cmtcus7gb002t01nsm8w8cfxu
 (Save this ID - you 'll use it throughout the notebook)


In [68]:
print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: processing
   Status: processing
   Status: completed

✅ Tree index ready!


In [69]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 1

🌲 Raw tree (first node):
{
  "title": "Vinod Sagar - Detailed CV Summary",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "# Vinod Sagar - Detailed CV Summary\n\nAI Engineer | Generative AI | Agentic AI | RAG Pipelines | LLM Systems\n",
  "text": "# Vinod Sagar - Detailed CV Summary\n\nAI Engineer | Generative AI | Agentic AI | RAG Pipelines | LLM Systems\n",
  "nodes": [
    {
      "title": "Profile Summary",
      "node_id": "0001",
      "page_index": 1,
      "summary": "## Profile Summary\n\nVinod Sagar is an AI Engineer based in Bengaluru with a strong focus on Generative AI, Agentic AI, LLM systems, and RAG pipelines. His profile highlights practical experience in building production-grade AI systems using LangGraph, LangChain, MCP, Microsoft Foundry, FastAPI, Django, vector databases, and AWS-based deployments.\n\nHe combines a traditional software engineering background in Python, Django, APIs, and data systems with modern AI engineering 

In [70]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Vinod Sagar - Detailed CV Summary  (p.1)
  └─ [0001] Profile Summary  (p.1)
  └─ [0002] Core AI Skills  (p.1)
  └─ [0003] Technical Stack  (p.1)
  └─ [0004] Current Role - Tap Health  (p.1)
  └─ [0005] Project Experience  (p.1)
  └─ [0006] Previous Experience  (p.1)
  └─ [0007] Career Break  (p.2)
  └─ [0008] Compensation, Availability, and Education  (p.2)


In [71]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 9
   Each node = one retrievable section of the document


In [72]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [73]:
def llm_tree_search(query: str, tree: list) -> dict:
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    compressed_tree = compress(tree)

    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = llm.invoke(prompt)
    return json.loads(response.content) # type: ignore

In [74]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "name of the person"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: name of the person

🧠 LLM Reasoning:
The query asks for the name of the person. The document title (node 0000) explicitly contains the name "Vinod Sagar". Additionally, the Profile Summary section (node 0001) starts with "Vinod Sagar is an AI Engineer..." which also provides the name. These two nodes are the most direct sources for the answer.

🎯 Selected Node IDs: ['0000', '0001']


In [75]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [76]:
from langchain_groq import ChatGroq

def generate_answer(query: str, nodes: list, model: str = "qwen/qwen3.8-27b") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    llm = ChatGroq(model=model, temperature=0)
    response = llm.invoke(prompt)
    
    return response.content # type: ignore

In [77]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [ ]:
# # ── Run the full pipeline ────────────────────────────────────────────────────
# answer = vectorless_rag(
#     query="What are the skills covered in this cv?",
#     tree=pageindex_tree
# )

🔍 Query: What are the skills covered in this cv?

🧠 Reasoning: The query asks for the skills covered in the CV. The sections that explicitly enumerate skills are the 'Core AI Skills' (node 0002) and the 'Technical Stack' (node 0003), which list specific AI, progr...
🎯 Retrieved node IDs: ['0002', '0003']
📄 Sections found: ['Core AI Skills', 'Technical Stack']

📝 Answer:
The CV covers the following skills:

*   **Core AI Skills**: Expertise in Agentic AI frameworks and protocols, including LangGraph workflows, LangChain agents, retrievers, vector stores, LCEL, callback systems, and MCP-based client-server architecture (Core AI Skills, Page 1).
*   **AI Engineering**: LLM gateways, AI guardrails, PII redaction, Human-in-the-Loop validation, LangSmith-based evaluation, hallucination detection, golden dataset regression testing, model routing, fallback model handling, and cost and latency optimization (Core AI Skills, Page 1).
*   **LLM Ecosystems**: Experience with OpenAI GPT models, Meta

In [ ]:
# ── Test with multiple queries ───────────────────────────────────────────────
# test_queries = [
#     "Why there was a GAp? and what you did?",
#     "What salary expected?",
#     "when can you join?",
# ]

# for q in test_queries:
#     print()
#     ans = vectorless_rag(q, pageindex_tree, verbose=False)
#     print(f"Q: {q}")
#     print(f"A: {ans[:300]}...")
#     print("-" * 55)


Q: Why there was a GAp? and what you did?
A: The gap occurred due to personal reasons (Career Break, Page 2). During this period, Vinod upgraded his skills in Generative AI, Agentic AI, RAG architecture, LLM application development, prompt engineering, vector databases, LangChain, LangGraph, FastAPI, MCP, Microsoft Foundry, Pinecone, AWS Bedro...
-------------------------------------------------------

Q: What salary expected?
A: The expected salary is 30 LPA (Compensation, Availability, and Education, Page 2)....
-------------------------------------------------------

Q: when can you join?
A: He can join within 15 days (Compensation, Availability, and Education, Page 2)....
-------------------------------------------------------


In [81]:
while True:
    query= input("ASK: ")
    if query == "quit":
        break

    print()
    ans = vectorless_rag(query, pageindex_tree, verbose=False)
    print(f"A: {ans[:300]}...")


A: Based on the provided context, the previous companies and organizations are:

*   **HCL Technologies** (Software Engineer) (Previous Experience, Page 1)
*   **Google India** (Python development) (Previous Experience, Page 1)
*   **Smith & Nephew** (Data engineering) (Previous Experience, Page 1)
*  ...

A: The name is Vinod Sagar ("Vinod Sagar - Detailed CV Summary", Page 1)....

A: Yes, Vinod worked at Mercedes-Benz R&D; India from August 2013 to July 2018 as a CAD Customization Engineer, focusing on CAD automation using VB.NET (Previous Experience, Page 1)....

A: Vinod Sagar worked at Volvo from August 2013 to July 2018 as a CAD Customization Engineer, focusing on CAD automation using VB.NET (Previous Experience, Page 1)....

A: ⚠️ No relevant sections found in the document....

A: Yes, the individual has worked on AI guardrails. This is explicitly listed as part of his AI engineering skill set, which includes "AI guardrails" and "PII redaction" (Core AI Skills, Page 1). Additio